In [1]:
%load_ext autoreload
%autoreload 2

determine keypoint visibility by rendering

In [2]:
import os 
import numpy as np

from pytorch3d.renderer import PerspectiveCameras, TexturesVertex
from pytorch3d.structures import Meshes

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import smplx 
import pytorch_lightning as pl
from einops import rearrange

from pytorch3d.structures import Meshes
from pytorch3d.renderer import (
    FoVPerspectiveCameras,
    PerspectiveCameras,
    RasterizationSettings,
    MeshRenderer,
    MeshRasterizer,
    TexturesVertex
)
from pytorch3d.renderer.blending import hard_rgb_blend, BlendParams



class SimpleShader(nn.Module):
    def __init__(self, device="cpu", blend_params=None):
        super().__init__()
        self.blend_params = blend_params if blend_params is not None else BlendParams()

    def forward(self, fragments, meshes, **kwargs) -> torch.Tensor:
        blend_params = kwargs.get("blend_params", self.blend_params)
        texels = meshes.sample_textures(fragments)
        
        # If we have more than 3 channels, return all channels
        if texels.shape[-1] > 3:
            # Use the alpha channel from fragments for masking
            alpha = fragments.zbuf[..., 0] > -1

            # Expand alpha to match number of channels
            alpha = alpha.unsqueeze(-1).unsqueeze(-1) # no need to expand as texels as same alpha for all channels

            # Return all channels with alpha mask
            return torch.cat([texels, alpha], dim=-1).squeeze() # (N, H, W, C+1)
        
        # For RGB textures, use standard blending
        images = hard_rgb_blend(texels, fragments, blend_params)
        return images  # (N, H, W, 3) RGBA image for RGB, or (N, H, W, C+1) for multi-channel
    

class FeatureRenderer(pl.LightningModule):
    def __init__(self, image_size=(256, 192)):
        super().__init__()
        self.image_size = image_size

        raster_settings = RasterizationSettings(
            image_size=self.image_size,
            blur_radius=0.0,
            faces_per_pixel=1,
            bin_size=None,
            max_faces_per_bin=160000 # Otherwise overflows 
        )
        
        self.renderer = MeshRenderer(
            rasterizer=MeshRasterizer(
                raster_settings=raster_settings
            ),
            # shader=SimpleShader()
        )
        # self.register_buffer('faces', smpl_faces)

        self._set_cameras(PerspectiveCameras().to(self.device))
    

    def forward(self, mesh, **kwargs):

        images = self.renderer(mesh)

        # Get fragments to determine visible parts of the mesh
        # fragments = self.renderer.rasterizer(mesh)
        
        # Extract visible mesh parts
        # visible_faces = self._extract_visible_faces(mesh, fragments)

        ret = {
            'maps': images[..., :-1],  # All channels except the last (alpha)
            'mask': images[..., -1],   # Last channel is the alpha/mask
            # 'visible_faces': visible_faces  # Indices of visible faces in original mesh
        }

        return ret
    

    def _set_cameras(self, cameras):
        self.renderer.rasterizer.cameras = cameras
        self.renderer.shader.cameras = cameras

    def _extract_visible_faces(self, mesh, fragments):

        # Get the face indices that are visible (have valid depth values)
        visible_faces = fragments.pix_to_face[..., 0]  # (N, H, W)
        valid_mask = visible_faces >= 0  # Faces with valid depth
        
        # Get unique face indices that are visible
        unique_visible_faces = torch.unique(visible_faces[valid_mask])
        
        # Remove the -1 (invalid face) index if present
        unique_visible_faces = unique_visible_faces[unique_visible_faces >= 0]
        
        return unique_visible_faces
    


def project(points, cam_trans, cam_int):
    points = points + cam_trans
    projected_points = points / points[..., -1].unsqueeze(-1)
    projected_points = torch.einsum("bij, bkj->bki", cam_int, projected_points)
    return projected_points


In [4]:
DATA_BASE_PATH = "/scratches/columbo2/cq244/BEDLAM/data/"
NPZ_PATH = os.path.join(DATA_BASE_PATH, "training_labels/all_npz_12_training_extra_mhr/20221010_3_1000_batch01hand_6fps.npz")
IMAGE_DIR = os.path.join(DATA_BASE_PATH, "training_images/20221010_3_1000_batch01hand_6fps")
MHR_MODEL_PATH = "/scratches/columbo2/cq244/sam-3d-body/checkpoints/sam-3d-body-dinov3/assets/mhr_model.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = np.load(NPZ_PATH)
print(list(data.keys()))
print(data['imgname'][:2])

print(f"identity_coeffs: {data['identity_coeffs'].shape}")
print(f"lbs_model_params: {data['lbs_model_params'].shape}")
print(f"face_expr_coeffs: {data['face_expr_coeffs'].shape}")


mhr_model = torch.jit.load(
    MHR_MODEL_PATH,
    map_location=("cuda" if torch.cuda.is_available() else "cpu"),
)

print(mhr_model)

chunk_size = 1024
vertices, joints3d = [], []
with torch.no_grad():
    num_samples = data['identity_coeffs'].shape[0]
    for i in range(0, num_samples, chunk_size):
        end_idx = min(i + chunk_size, num_samples)
        mhr_input = {
            'identity_coeffs': torch.tensor(data['identity_coeffs'][i:end_idx], dtype=torch.float32, device=device),
            'model_parameters': torch.tensor(data['lbs_model_params'][i:end_idx], dtype=torch.float32, device=device),
            'face_expr_coeffs': torch.tensor(data['face_expr_coeffs'][i:end_idx], dtype=torch.float32, device=device),
        }
        verts, skeleton = mhr_model(**mhr_input)
        j3d = skeleton[:, :, :3]
        vertices.append(verts.cpu().detach().numpy())
        joints3d.append(j3d.cpu().detach().numpy())
        
        if i > 5000:
            break 

vertices = np.concatenate(vertices, axis=0)
joints3d = np.concatenate(joints3d, axis=0)

print(vertices.shape, joints3d.shape)



['imgname', 'center', 'scale', 'pose_cam', 'pose_world', 'shape', 'trans_cam', 'trans_world', 'gtkps', 'cam_int', 'cam_ext', 'gender', 'proj_verts', 'serno', 'lbs_model_params', 'identity_coeffs', 'face_expr_coeffs', 'mhr_keypoints_2d']
['seq_000000/seq_000000_0000.png' 'seq_000000/seq_000000_0005.png']
identity_coeffs: (89728, 45)
lbs_model_params: (89728, 204)
face_expr_coeffs: (89728, 72)
RecursiveScriptModule(
  original_name=MHRDemo
  (face_expressions_model): RecursiveScriptModule(original_name=BlendShapeBase)
  (pose_correctives_model): RecursiveScriptModule(
    original_name=MHRPoseCorrectivesModel
    (pose_dirs_predictor): RecursiveScriptModule(
      original_name=Sequential
      (0): RecursiveScriptModule(original_name=SparseLinear)
      (1): RecursiveScriptModule(original_name=ReLU)
      (2): RecursiveScriptModule(original_name=Linear)
    )
  )
  (character_torch): RecursiveScriptModule(
    original_name=Character
    (skeleton): RecursiveScriptModule(original_name=S

In [ ]:

if "PYOPENGL_PLATFORM" not in os.environ:
    os.environ["PYOPENGL_PLATFORM"] = "egl"
import pyrender
# from sam_3d_body.visualization.renderer import Renderer

ckpt = torch.load("/scratches/columbo2/cq244/sam-3d-body/checkpoints/sam-3d-body-dinov3/model.ckpt", weights_only=False)
print(type(ckpt))
for key in ckpt.keys():
    if 'faces' in key:
        print(key)

faces = ckpt['head_pose.faces'].cpu().detach().numpy()
print(faces.shape)
# del ckpt
# import gc
# gc.collect()
# torch.cuda.empty_cache()






<class 'dict'>
head_pose.faces
head_pose_hand.faces
(36874, 3)


: 